In [2]:
!pip install torchmetrics==0.11.4 torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchtext==0.14.1 torchaudio==0.13.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu117 --no-cache-dir --quiet
!pip install h5py --quiet
!pip install scikit-image --quiet
!pip install boto3 --quiet
!pip install "numpy<1.24" --quiet
!pip install skm-tea -q
!pip install 'meddlr[all]' -q
!pip install intensity-normalization --quiet

In [3]:
!pip install albumentations -q

In [4]:
!pip install timm -U -q
# !pip install urllib3==1.25.4 -q
# !pip install boto3 -q #-U

In [5]:
import albumentations as A
import albumentations.augmentations.functional as F

import copy
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import timm
import torch
import torchvision as tv
import torch.nn as nn
import json
import boto3
from io import BytesIO
from io import StringIO
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import GroupKFold, KFold
import gc
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix,
    f1_score, 
    precision_score, 
    recall_score,
    roc_auc_score,
    balanced_accuracy_score,
    average_precision_score
)
from torch.cuda.amp import GradScaler
from torch.cuda.amp import autocast
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time
import seaborn as sns
import constants
from  utilities import list_all_objects_in_s3, from_s3_npy, read_file_from_s3, write_file_to_s3

In [6]:
from sagemaker import get_execution_role, session
from sagemaker.s3 import S3Downloader, S3Uploader

region = boto3.Session().region_name
role = get_execution_role()
sm_session = session.Session(boto3.Session())
sm = boto3.Session().client("sagemaker")
sm_runtime = boto3.Session().client("sagemaker-runtime")

bucket = sm_session.default_bucket()
prefix = "sagemaker/DEMO-pathology-detection"

In [8]:
# !pip list --format=freeze > requirements.txt

In [35]:
import os
import tarfile

if os.path.exists('model.tar.gz'):
    os.remove('model.tar.gz')
    
# Create 'model.tar.gz' and add 'model.pth', along with other files (e.g., inference.py)
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add("abnorm_efficientnetb1-f0.tph", arcname="abnorm_efficientnetb1-f0.tph") #removing this step as we are using google download
    tar.add('inference.py', arcname='code/inference.py')
    tar.add('constants.py', arcname='code/constants.py')
    tar.add('utilities.py', arcname='code/utilities.py')
    tar.add('requirements.txt', arcname='code/requirements.txt')

In [36]:
#upload model.tar.gz to S3
model_url = S3Uploader.upload(
    local_path="model.tar.gz", desired_s3_uri=f"s3://{bucket}/{prefix}"
)

model_url

's3://sagemaker-us-west-1-468914279846/sagemaker/DEMO-pathology-detection/model.tar.gz'

In [37]:
from sagemaker.image_uris import retrieve
from datetime import datetime

model_name = f"DEMO-pathology-predictionx1"
image_uri = retrieve("pytorch",
                     boto3.Session().region_name, version="1.13.1",
    py_version="py39",image_scope="inference", instance_type="ml.g4dn.4xlarge")
sm_session.create_model(
    name=model_name, role=role, container_defs={"Image": image_uri, "ModelDataUrl": model_url}
)

'DEMO-pathology-predictionx1'

In [38]:
from sagemaker.session import production_variant

variant1 = production_variant(
    model_name=model_name,
    instance_type="ml.g4dn.8xlarge",
    initial_instance_count=1,
    variant_name="VariantN1",
    initial_weight=1,
)

(variant1)

{'ModelName': 'DEMO-pathology-predictionx1',
 'VariantName': 'VariantN1',
 'InitialVariantWeight': 1,
 'InitialInstanceCount': 1,
 'InstanceType': 'ml.g4dn.8xlarge'}

In [39]:
endpoint_name = f"DEMO-pathology-predictionx-endpoint"
print(f"EndpointName={endpoint_name}")

sm_session.endpoint_from_production_variants(
    name=endpoint_name, production_variants=[variant1]
)

EndpointName=DEMO-pathology-predictionx-endpoint
----------!

'DEMO-pathology-predictionx-endpoint'

In [40]:
# Test endpoint
# Your input data as a Python dictionary
# put slice range : 0-80 and then rerun all below cells again with slice range 80-159
input_data = {'image_id': 'MTR_174' } #, 'slice_start_range':0 , 'slice_end_range':80
# Serialize the input data to JSON format
payload = json.dumps(input_data)

# Convert the JSON payload to bytes
payload = payload.encode('utf-8')
sm_runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json", Body=payload)

{'ResponseMetadata': {'RequestId': 'a3fb2e43-14df-44ef-a4e1-c470c97b25da',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a3fb2e43-14df-44ef-a4e1-c470c97b25da',
   'x-amzn-invoked-production-variant': 'VariantN1',
   'date': 'Sat, 29 Jul 2023 08:22:03 GMT',
   'content-type': 'application/json',
   'content-length': '149',
   'connection': 'keep-alive'},
  'RetryAttempts': 0},
 'ContentType': 'application/json',
 'InvokedProductionVariant': 'VariantN1',
 'Body': <botocore.response.StreamingBody at 0x7f6744878430>}

In [41]:
#check if sliceInfo and summary json is uploaded :: 
import utilities
utilities.list_all_objects_in_s3('stats-ui')

['stats-ui/pathology/',
 'stats-ui/pathology/sliceInfo/',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_1.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_10.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_100.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_101.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_102.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_103.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_104.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_105.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_106.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_107.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_108.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_109.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_11.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_110.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_111.json',
 'stats-ui/pathology/sliceInfo/MTR_001/MTR_001_112.json',
 'stats-ui/patholo

In [42]:
#check summary content 
import boto3
import json
def read_json_from_s3(bucket_name, file_key):
    """
    Read a JSON file from the specified AWS S3 bucket.

    Args:
        bucket_name (str): The name of the S3 bucket.
        file_key (str): The key (path) of the JSON file in the S3 bucket.

    Returns:
        dict: The JSON data read from the file.

    Raises:
        botocore.exceptions.ClientError: If the file does not exist or if there's an error reading it.
    """
    s3 = boto3.client('s3')
    response = s3.get_object(Bucket=bucket_name, Key=file_key)
    data = response['Body'].read().decode('utf-8')
    return json.loads(data)

x=read_json_from_s3('stats-ui','pathology/summary/MTR_174_pathology_summary.json')
print(x)

{'image_id': 'MTR_174', 'abnormality': [{'start_sl': 43, 'end_sl': 121, 'slice_count': 79}], 'summary': 'abnormality found on slice 43 to slice 121'}


In [3]:
# import boto3
# client = boto3.client("s3")

# def list_all_objects_in_s3(bucketName, prefixName=None):
#     """Get a list of objects in an S3 bucket path.
#     @params:
#     bucketName : s3 bucket 
#     prefixName : s3 prefix to bucket
#     returns : list of objects in s3 path
#     [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}"]
#     """
#     if prefixName:
#         resp = client.list_objects_v2(Bucket=bucketName, StartAfter=prefixName)
#     else:
#         resp = client.list_objects_v2(Bucket=bucketName)
#     list_of_files=[]
#     for obj in resp['Contents']:
#         files = bucketName+'/'+obj['Key']
#         if prefixName:
#             if prefixName in files:
#                 if prefixName!=files:
#                     list_of_files.append(files)
#         else:
#             list_of_files.append(files)
#     return list_of_files

# def from_s3_npy(s3_uri: str):
#     """
#     @params 
#     data : file content 
#     s3_uri : s3 file path along with name [example of S3 bucket with prefix f"s3://{bucketName}/{prefix}/{fileName.npy}"]
#     """
#     try:
#         bytes_ = BytesIO()
#         parsed_s3 = urlparse(s3_uri)
#         client.download_fileobj(
#             Fileobj=bytes_, Bucket=parsed_s3.netloc, Key=parsed_s3.path[1:]
#         )
#         bytes_.seek(0)
#         np_data=np.load(bytes_, allow_pickle=True)
    
#     except Exception as e: # work on python 3.x
#         print('Failed to read numpy array from S3: '+ str(e))
#     return np_data

# s3 = boto3.resource("s3")
# def read_file_from_s3(bucket, file):
#     try:
#         obj = s3.Object(bucket, file)
#         data = obj.get()["Body"]
        
#     except Exception as e: # work on python 3.x
#         print("Failed to read file from S3: "+ str(e))
#     return data.read().decode(encoding="utf-8")

In [73]:
# def write_file_to_s3(bucketName, fileContent, fileName, fileType=None, prefixName=None):
#     """
#         @param
#         bucketName : s3 bucket
#         fileContent : data to be written
#         fileName   : output file Name
#         fileType : optional; valid values : json (to write JSON file) 
#                                             dataframe (to write df as csv file)
#                                             None (any default file format)
#         prefixName : optional; s3 prefix to bucket
#     """
#     try:
#         if prefixName:
#             fileName=prefixName+'/'+fileName
#         if fileType=='json':
#             response = client.put_object(Bucket=bucketName, #'testshi'
#                                          Body=bytes(json.dumps(fileContent, default=str).encode('UTF-8')), #json.dumps(fileContent),  # '{"content": "bytes or seekable file-like object"}',
#                                          Key=fileName)  #'test.json' # 'Object key for which the PUT operation was initiated'
#         if fileType=='dataframe':
#             fileContent.to_csv('s3://'+bucketName+'/'+fileName+'.csv', index=False)
#         else:
#             response = client.put_object( Bucket=bucketName, #'testshi'
#                                          Body=fileContent, # '{"content": "bytes or seekable file-like object"}',
#                                          Key=fileName) #'test.json' # 'Object key for which the PUT operation was initiated'
#         print('Writing to S3 completed for ',fileName)
#     except Exception as e: # work on python 3.x
#         print('Failed to write numpy to S3: '+ str(e))
#     return

In [110]:
# def predict_lesion(x, model, device, verbose=False):

In [89]:
# model = load_model(LesionModel(BACKBONE), "abnorm_efficientnetb1-f0.tph").to(DEVICE)


In [90]:
# def make_prediction_abnorm_slice(bucketName, scan_id, sl):
#     img = from_s3_npy(bucketName + scan_id + "_" + str(sl) + ".npy")
#     img = torch.from_numpy(img).unsqueeze(0)
#     y_pred = predict_lesion(img, model, DEVICE, True)

In [5]:
# list_all_objects_in_s3('skm-dataset-echo1')

In [169]:
def make_prediction_abnorm_stack_ui(bucketName, scan_id):
    # bucketName = "s3://skm-dataset-echo1/"
    # scan_id = "MTR_173"
    
    metadata_summary = {}
    metadata_summary = {
        "image_id" : scan_id,
        "abnormality": [],
        "summary": ""
    }
    
    threshold = 0.8
    abnorm_start, abnorm_end, abnorm_count = -1, -1, 0

    for sl in range(1, 161):

        img = from_s3_npy(bucketName + scan_id + "_" + str(sl) + ".npy")
        img = torch.from_numpy(img).unsqueeze(0)
        y_pred = predict_lesion(img, model, DEVICE, False)
        if y_pred[0] >= threshold:
            if abnorm_start == -1:
                abnorm_start = sl
                abnorm_end = sl
            else:
                abnorm_end = sl

    if abnorm_start == -1:
        metadata_summary["abnormality"].append({"start_sl": None, "end_sl": None, "slice_count": 0})
        metadata_summary["summary"] = f"no pathology found"
    else:
        metadata_summary["abnormality"].append({"start_sl": abnorm_start, "end_sl": abnorm_end, "slice_count": abnorm_end - abnorm_start + 1})
        metadata_summary["summary"] = f"abnormality found on slice {abnorm_start} to slice {abnorm_end}"

    metadata_summary_json = json.dumps(metadata_summary)
    outSummaryFileName = scan_id + "_pathology_summary.json"
    write_file_to_s3("stats-ui", metadata_summary_json, outSummaryFileName, "json", "pathology/summary")

In [170]:
t0 = time.time()

make_prediction_abnorm_stack_ui("s3://skm-dataset-echo1/", "MTR_005")

print(f"prediction time per image file: {time.time() - t0}")

Writing to S3 completed for  pathology/summary/MTR_005_pathology_summary.json
prediction time per image file: 21.753692388534546


In [86]:
test = pd.read_csv(StringIO(read_file_from_s3("master-annotation-files", "test_annotations.csv")))
test.scan_id.unique()

array(['MTR_173', 'MTR_176', 'MTR_178', 'MTR_188', 'MTR_196', 'MTR_198',
       'MTR_219', 'MTR_221', 'MTR_218', 'MTR_199', 'MTR_224', 'MTR_227',
       'MTR_235', 'MTR_237', 'MTR_240', 'MTR_241', 'MTR_244', 'MTR_248',
       'MTR_005', 'MTR_006', 'MTR_030', 'MTR_034', 'MTR_048', 'MTR_052',
       'MTR_065', 'MTR_066', 'MTR_079', 'MTR_080', 'MTR_096', 'MTR_099',
       'MTR_120', 'MTR_126', 'MTR_144', 'MTR_156', 'MTR_158'],
      dtype=object)

In [160]:
test[test["scan_id"] == "MTR_005"].sort_values(by="sl")

,image_id,category_id,tissue_id,bbox,confidence,labeler,category,supercat_id,supercategory,tissues,file_name,msp_id,msp_file_name,scan_id,subject_id,split,sl,2d_bbox,yolo_bbox
2389,1,13,5,"[128.0, 80.0, 36.0, 95.0, 47.0, 60.0]",3.0,2,Cartilage Lesion (2A),3,Cartilage Lesion,Patellar Cartilage,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,36,"[80.0, 128.0, 47.0, 95.0]","[0.2021484375, 0.3427734375, 0.091796875, 0.18..."
2390,1,13,5,"[128.0, 80.0, 36.0, 95.0, 47.0, 60.0]",3.0,2,Cartilage Lesion (2A),3,Cartilage Lesion,Patellar Cartilage,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,37,"[80.0, 128.0, 47.0, 95.0]","[0.2021484375, 0.3427734375, 0.091796875, 0.18..."
2391,1,13,5,"[128.0, 80.0, 36.0, 95.0, 47.0, 60.0]",3.0,2,Cartilage Lesion (2A),3,Cartilage Lesion,Patellar Cartilage,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,38,"[80.0, 128.0, 47.0, 95.0]","[0.2021484375, 0.3427734375, 0.091796875, 0.18..."
2392,1,13,5,"[128.0, 80.0, 36.0, 95.0, 47.0, 60.0]",3.0,2,Cartilage Lesion (2A),3,Cartilage Lesion,Patellar Cartilage,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,39,"[80.0, 128.0, 47.0, 95.0]","[0.2021484375, 0.3427734375, 0.091796875, 0.18..."
2393,1,13,5,"[128.0, 80.0, 36.0, 95.0, 47.0, 60.0]",3.0,2,Cartilage Lesion (2A),3,Cartilage Lesion,Patellar Cartilage,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,40,"[80.0, 128.0, 47.0, 95.0]","[0.2021484375, 0.3427734375, 0.091796875, 0.18..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2384,1,5,1,"[298.0, 121.0, 90.0, 29.0, 215.0, 33.0]",2.0,2,Meniscal Tear (Oblique),1,Meniscal Tear,Meniscus,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,118,"[121.0, 298.0, 215.0, 29.0]","[0.4462890625, 0.6103515625, 0.419921875, 0.05..."
2385,1,5,1,"[298.0, 121.0, 90.0, 29.0, 215.0, 33.0]",2.0,2,Meniscal Tear (Oblique),1,Meniscal Tear,Meniscus,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,119,"[121.0, 298.0, 215.0, 29.0]","[0.4462890625, 0.6103515625, 0.419921875, 0.05..."
2386,1,5,1,"[298.0, 121.0, 90.0, 29.0, 215.0, 33.0]",2.0,2,Meniscal Tear (Oblique),1,Meniscal Tear,Meniscus,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,120,"[121.0, 298.0, 215.0, 29.0]","[0.4462890625, 0.6103515625, 0.419921875, 0.05..."
2387,1,5,1,"[298.0, 121.0, 90.0, 29.0, 215.0, 33.0]",2.0,2,Meniscal Tear (Oblique),1,Meniscal Tear,Meniscus,MTR_005.h5,0000099_V00,0000099_V00.h5,MTR_005,99,test,121,"[121.0, 298.0, 215.0, 29.0]","[0.4462890625, 0.6103515625, 0.419921875, 0.05..."
